# Advanced Digital Image Processing — DIPA Project v3
## Complete Pipeline Notebook

**Pipeline Steps:**
1. Setup & Image Generation
2. Noise Injection
3. Spatial & Frequency Filtering (Restoration)
4. Filter Comparison (MSE / PSNR)
5. Segmentation
6. Morphological Processing
7. Feature Extraction
8. PCA + Classification


## 0. Setup & Imports

In [64]:
import os, sys
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tabulate import tabulate

# Make sure project modules are importable
PROJECT_DIR = r"C:\Users\Asus\Desktop"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

matplotlib.rcParams["figure.dpi"] = 120
print("Working directory:", os.getcwd())


Working directory: C:\Users\Asus\Desktop


## 1. Generate / Load Images

Loads real images from `images/raw/` if present, otherwise synthesises them.
Also regenerates binary masks via Otsu thresholding.


In [65]:
from generate_images import generate_all, GENERATORS
from pipeline.noise import DEFAULT_NOISE_CONFIG

image_paths = generate_all()

# Load all images and masks into dicts
images, masks = {}, {}
for name in GENERATORS:
    images[name] = cv2.imread(image_paths[name]["image"],  cv2.IMREAD_GRAYSCALE)
    masks[name]  = cv2.imread(image_paths[name]["mask"],   cv2.IMREAD_GRAYSCALE)

# Quick preview (one per class)
class_reps = ["circle", "wrench", "hexagon", "leaf", "washer"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, name in zip(axes, class_reps):
    ax.imshow(images[name], cmap="gray")
    ax.set_title(name.capitalize(), fontsize=11)
    ax.axis("off")
plt.suptitle("Original Images (one per class)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


  [✓] circle     <- real image  (images\raw\circle.png)
  [✓] wrench     <- real image  (images\raw\wrench.png)
  [✓] hexagon    <- real image  (images\raw\hexagon.png)
  [✓] leaf       <- real image  (images\raw\leaf.png)
  [✓] washer     <- real image  (images\raw\washer.png)

Ready: 5 images + masks.


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\85742422.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Noise Injection

Each object class is paired with a specific noise type:

| Object  | Noise type   | Parameters          |
|---------|-------------|---------------------|
| Circle  | Salt & Pepper | density = 0.05    |
| Wrench  | Gaussian    | mean=0, σ=25        |
| Hexagon | Periodic    | freq=30, amp=40     |
| Leaf    | Gamma       | shape=5, scale=5    |
| Washer  | Uniform     | low=-30, high=30    |


In [66]:
from pipeline.noise import apply_noise

noisy_images = {}
for name in class_reps:
    cfg = DEFAULT_NOISE_CONFIG[name]
    noisy_images[name] = apply_noise(images[name], cfg["type"], **cfg["params"])

# Visualise original vs noisy
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for col, name in enumerate(class_reps):
    axes[0, col].imshow(images[name],      cmap="gray"); axes[0, col].set_title(f"{name} — original"); axes[0, col].axis("off")
    axes[1, col].imshow(noisy_images[name], cmap="gray")
    cfg = DEFAULT_NOISE_CONFIG[name]
    axes[1, col].set_title(f"+ {cfg['type']} noise"); axes[1, col].axis("off")
plt.suptitle("Original vs Noisy", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\2689230579.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Image Restoration

### 3a. Spatial-domain filters
Mean, Gaussian, Median (3×3 and 5×5), Laplacian sharpening, Sobel sharpening.

### 3b. Frequency-domain filters
Ideal and Butterworth Low-Pass / High-Pass filters at various cut-offs.
Notch filter for periodic noise (hexagon).

### 3c. Enhancement
Histogram Equalization and CLAHE applied to the best spatial result.


In [67]:
from pipeline.restoration       import apply_all_spatial_filters, histogram_equalization, clahe
from pipeline.frequency_filters import (apply_all_frequency_filters, apply_fft_filter,
                                         notch_reject, ideal_lowpass, butterworth_lowpass)
from pipeline.comparator        import compute_psnr, compute_mse

all_filter_results = {}   # {obj_name: {filter_name: image}}

for name in class_reps:
    noisy      = noisy_images[name]
    noise_cfg  = DEFAULT_NOISE_CONFIG[name]

    spatial = apply_all_spatial_filters(noisy)
    freq    = apply_all_frequency_filters(noisy)

    # Notch filter only for periodic noise
    if noise_cfg["type"] == "periodic":
        freq_param = noise_cfg["params"].get("frequency", 30)
        for bw in [5, 10, 15]:
            nmask = notch_reject(noisy.shape, center_freq=freq_param, bandwidth=bw)
            freq[f"Notch BW={bw}"] = apply_fft_filter(noisy, nmask)

    combined = {**spatial, **freq}

    # CLAHE / HistEq on best spatial
    best_sp = max(spatial, key=lambda k: compute_psnr(images[name], spatial[k]))
    combined[f"{best_sp} + HistEq"] = histogram_equalization(spatial[best_sp])
    combined[f"{best_sp} + CLAHE"]  = clahe(spatial[best_sp])

    all_filter_results[name] = combined

# Show a few spatial results for 'circle'
demo = "circle"
demo_filters = ["Mean 3×3", "Median 3×3", "Median 5×5", "Gaussian 5×5"]
fig, axes = plt.subplots(1, len(demo_filters)+1, figsize=(18, 3.5))
axes[0].imshow(noisy_images[demo], cmap="gray"); axes[0].set_title("Noisy"); axes[0].axis("off")
for ax, fname in zip(axes[1:], demo_filters):
    ax.imshow(all_filter_results[demo][fname], cmap="gray")
    ax.set_title(fname, fontsize=9); ax.axis("off")
plt.suptitle(f"{demo.capitalize()} — Spatial Filter Comparison", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\2614892631.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Filter Comparison (MSE / PSNR)

Rank all filters by PSNR (higher = better quality).


In [68]:
from pipeline.comparator import evaluate_all_filters, compute_difference_map, generate_comparison_figure

best_restored = {}

for name in class_reps:
    original  = images[name]
    noisy     = noisy_images[name]
    filters   = all_filter_results[name]

    eval_table = evaluate_all_filters(original, noisy, filters)

    # Top 5
    print(f"\n{'='*55}")
    print(f"  {name.upper()} — Top-5 Filters by PSNR")
    print(f"{'='*55}")
    print(tabulate(eval_table[:5], headers="keys", floatfmt=".2f", tablefmt="github"))

    best_name = eval_table[0]["filter"]
    best_restored[name] = filters.get(best_name, noisy)

    # Comparison figure
    diff = compute_difference_map(original, best_restored[name])
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (img, title) in zip(axes, [
            (original, "Original"),
            (noisy,    f"Noisy ({DEFAULT_NOISE_CONFIG[name]['type']})"),
            (best_restored[name], f"Restored\n{best_name}"),
            (diff, "Difference Map")]):
        ax.imshow(img, cmap="gray"); ax.set_title(title, fontsize=9); ax.axis("off")
    psnr_val = eval_table[0]["psnr"]
    mse_val  = eval_table[0]["mse"]
    fig.text(0.5, 0.01, f"MSE = {mse_val:.2f}  |  PSNR = {psnr_val:.2f} dB",
             ha="center", fontsize=10, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.6))
    plt.suptitle(f"{name.capitalize()} — Best Filter: {best_name}", fontsize=11, fontweight="bold")
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.show()



  CIRCLE — Top-5 Filters by PSNR
| filter               |    mse |   psnr |
|----------------------|--------|--------|
| Median 3×3           |  33.07 |  32.94 |
| Median 5×5           |  95.57 |  28.33 |
| Butterworth LP D0=60 | 217.12 |  24.76 |
| Gaussian 5×5         | 220.92 |  24.69 |
| Mean 5×5             | 241.89 |  24.29 |


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\1612324110.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



  WRENCH — Top-5 Filters by PSNR
| filter               |   mse |   psnr |
|----------------------|-------|--------|
| Butterworth LP D0=60 | 53.96 |  30.81 |
| Gaussian 5×5         | 55.93 |  30.65 |
| Mean 5×5             | 58.26 |  30.48 |
| Ideal LP D0=60       | 66.74 |  29.89 |
| Median 5×5           | 68.55 |  29.77 |

  HEXAGON — Top-5 Filters by PSNR
| filter               |    mse |   psnr |
|----------------------|--------|--------|
| Notch BW=5           |   2.13 |  44.84 |
| Notch BW=10          |  29.98 |  33.36 |
| Butterworth LP D0=30 | 223.04 |  24.65 |
| Mean 5×5             | 615.52 |  20.24 |
| Gaussian 5×5         | 662.54 |  19.92 |

  LEAF — Top-5 Filters by PSNR
| filter               |    mse |   psnr |
|----------------------|--------|--------|
| Butterworth LP D0=30 | 230.88 |  24.50 |
| Ideal LP D0=30       | 242.95 |  24.28 |
| Butterworth LP D0=60 | 293.15 |  23.46 |
| Mean 5×5             | 308.61 |  23.24 |
| Ideal LP D0=60       | 327.55 |  22.98 |

  

## 5. Edge Detection

Sobel (gradient magnitude) and Laplacian applied to the best-restored images.


In [69]:
from pipeline.edge_detection import sobel_edges, laplacian_edges

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for col, name in enumerate(class_reps):
    restored = best_restored[name]
    mag, gx, gy = sobel_edges(restored)
    lap        = laplacian_edges(restored)

    axes[0, col].imshow(restored, cmap="gray"); axes[0, col].set_title(name.capitalize()); axes[0, col].axis("off")
    axes[1, col].imshow(mag,      cmap="hot");  axes[1, col].set_title("Sobel mag");       axes[1, col].axis("off")
    axes[2, col].imshow(lap,      cmap="hot");  axes[2, col].set_title("Laplacian");        axes[2, col].axis("off")

plt.suptitle("Edge Detection (Sobel & Laplacian)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\1221855033.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Segmentation

Three methods evaluated against the ground-truth mask using **IoU**:
- Global thresholding (best threshold found by grid search)
- Otsu's automatic thresholding
- Adaptive (Gaussian-weighted local) thresholding


In [70]:
from pipeline.segmentation import evaluate_segmentation, select_best_segmentation

best_seg_masks  = {}
best_seg_methods = {}

print(f"{'Object':<10} {'Global IoU':>12} {'Otsu IoU':>10} {'Adaptive IoU':>14} {'Best':>10}")
print("-" * 52)

for name in class_reps:
    seg = evaluate_segmentation(best_restored[name], masks[name])
    method = select_best_segmentation(seg)
    best_seg_masks[name]   = seg[method]["mask"]
    best_seg_methods[name] = method
    g  = seg["Global"]["iou"]
    o  = seg["Otsu"]["iou"]
    a  = seg["Adaptive"]["iou"]
    print(f"{name:<10} {g:>12.4f} {o:>10.4f} {a:>14.4f} {method:>10}")

# Visualise
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for col, name in enumerate(class_reps):
    seg = evaluate_segmentation(best_restored[name], masks[name])
    axes[0, col].imshow(best_restored[name],  cmap="gray"); axes[0, col].set_title(name.capitalize()); axes[0, col].axis("off")
    axes[1, col].imshow(seg["Otsu"]["mask"],  cmap="gray"); axes[1, col].set_title(f"Otsu (IoU={seg['Otsu']['iou']:.3f})"); axes[1, col].axis("off")
    axes[2, col].imshow(best_seg_masks[name], cmap="gray"); axes[2, col].set_title(f"Best: {best_seg_methods[name]}"); axes[2, col].axis("off")
plt.suptitle("Segmentation Results", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


Object       Global IoU   Otsu IoU   Adaptive IoU       Best
----------------------------------------------------
circle           0.9879     0.9890         0.5894       Otsu
wrench           0.9956     0.9957         0.7337       Otsu
hexagon          0.9969     0.9985         0.9537       Otsu
leaf             0.9944     0.9943         0.7806     Global
washer           0.7482     0.7721         0.4451       Otsu


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\71862735.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7. Morphological Processing

Tries Opening, Closing, Erosion, Dilation with rect/ellipse/cross kernels (3×3 to 7×7).
The combination yielding the highest IoU vs ground-truth is selected.


In [71]:
from pipeline.morphology import evaluate_morphology, boundary_extraction

best_morph_masks = {}

for name in class_reps:
    morph_eval = evaluate_morphology(best_seg_masks[name], masks[name])
    best_morph_masks[name] = morph_eval.get("best_mask", best_seg_masks[name])

    print(f"{name:<10}  best op: {morph_eval['best_operation']:<10} "
          f"kernel: {morph_eval['best_kernel_shape']}-{morph_eval['best_kernel_size']}  "
          f"IoU: {morph_eval.get('best_iou', 0):.4f}")

# Visualise: seg mask → best morph → boundary
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for col, name in enumerate(class_reps):
    boundary = boundary_extraction(best_morph_masks[name])
    axes[0, col].imshow(best_seg_masks[name],  cmap="gray"); axes[0, col].set_title(name.capitalize()); axes[0, col].axis("off")
    axes[1, col].imshow(best_morph_masks[name],cmap="gray"); axes[1, col].set_title("After Morphology"); axes[1, col].axis("off")
    axes[2, col].imshow(boundary,              cmap="gray"); axes[2, col].set_title("Boundary"); axes[2, col].axis("off")
plt.suptitle("Morphological Processing & Boundary Extraction", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


circle      best op: Closing    kernel: ellipse-3  IoU: 0.0000
wrench      best op: Closing    kernel: ellipse-3  IoU: 0.0000
hexagon     best op: Closing    kernel: ellipse-3  IoU: 0.0000
leaf        best op: Opening    kernel: cross-7  IoU: 0.0000
washer      best op: Closing    kernel: ellipse-3  IoU: 0.0000


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\2894294653.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 8. Feature Extraction

Seven geometric features computed from each binary mask:

| Feature | Description |
|---------|-------------|
| Area | Contour area in pixels² |
| Perimeter | Contour perimeter in pixels |
| Circularity | 4π·area / perimeter² ∈ (0,1] |
| Aspect Ratio | Bounding-box width / height |
| Extent | Area / bounding-box area |
| Solidity | Area / convex-hull area |
| Num Holes | Internal contours (holes) |


In [72]:
from pipeline.features import extract_features, features_to_table, save_features_csv

all_features = {}
for name in class_reps:
    feat = extract_features(best_morph_masks[name])
    all_features[name] = feat

headers, rows = features_to_table(all_features)
print(tabulate(rows, headers=headers, tablefmt="github", floatfmt=".4f"))

os.makedirs("output/tables", exist_ok=True)
save_features_csv(all_features, "output/tables/features.csv")


| Object   |        Area |   Perimeter |   Circularity |   Aspect Ratio |   Extent |   Solidity |   Holes |
|----------|-------------|-------------|---------------|----------------|----------|------------|---------|
| Circle   | 261121.0000 |   2044.0000 |        0.7854 |         1.0000 |   0.9961 |     1.0000 |       1 |
| Wrench   | 255467.5000 |   2356.5800 |        0.5781 |         1.0000 |   0.9745 |     0.9783 |       1 |
| Hexagon  | 261121.0000 |   2044.0000 |        0.7854 |         1.0000 |   0.9961 |     1.0000 |       6 |
| Leaf     | 261121.0000 |   2044.0000 |        0.7854 |         1.0000 |   0.9961 |     1.0000 |       1 |
| Washer   |  95955.5000 |   4860.7700 |        0.0510 |         1.4884 |   0.5448 |     0.7164 |     455 |
  [+] Features saved to output/tables/features.csv


## 9. PCA + Classification

1. Augment features with Gaussian jitter (×15 per sample)
2. Standardise (zero mean, unit variance)
3. PCA → 2 principal components
4. Minimum Distance Classifier (nearest class centroid)


In [73]:
from pipeline.classification import run_classification

os.makedirs("output", exist_ok=True)
clf_output = run_classification(all_features, save_dir="output")
print(clf_output["report"])
print(f"\nOverall accuracy: {clf_output['accuracy']*100:.1f}%")


  [+] PCA scatter saved to output\pca\pca_scatter.png
Classification Report
Object          True         Predicted    Correct
--------------------------------------------------
Circle          Circular     Irregular    ✗
Wrench          Elongated    Elongated    ✓
Hexagon         Polygonal    Circular     ✗
Leaf            Irregular    Irregular    ✓
Washer          Hollow       Hollow       ✓
--------------------------------------------------
Accuracy: 3/5 = 60.0%
Classification Report
Object          True         Predicted    Correct
--------------------------------------------------
Circle          Circular     Irregular    ✗
Wrench          Elongated    Elongated    ✓
Hexagon         Polygonal    Circular     ✗
Leaf            Irregular    Irregular    ✓
Washer          Hollow       Hollow       ✓
--------------------------------------------------
Accuracy: 3/5 = 60.0%

Overall accuracy: 60.0%


In [74]:
# Show PCA scatter plot
pca_img = cv2.imread(clf_output["pca_path"])
if pca_img is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(pca_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("PCA Scatter Plot", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


C:\Users\Asus\AppData\Local\Temp\ipykernel_14032\3091457295.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Pipeline Summary

Final results table for all 5 objects.


In [75]:
from pipeline.comparator import compute_mse, compute_psnr
from pipeline.segmentation import evaluate_segmentation, select_best_segmentation
from pipeline.morphology import evaluate_morphology
from pipeline.classification import CLASS_LABELS

summary_rows = []
for name in class_reps:
    noise_type   = DEFAULT_NOISE_CONFIG[name]["type"]
    psnr_val     = compute_psnr(images[name], best_restored[name])
    mse_val      = compute_mse(images[name],  best_restored[name])
    seg          = evaluate_segmentation(best_restored[name], masks[name])
    best_seg     = select_best_segmentation(seg)
    morph        = evaluate_morphology(best_seg_masks[name], masks[name])
    pred_class   = CLASS_LABELS.get(name, "?")
    summary_rows.append([
        name.capitalize(),
        noise_type,
        f"{psnr_val:.2f} dB",
        f"{mse_val:.2f}",
        best_seg,
        f"{seg[best_seg]['iou']:.4f}",
        f"{morph['best_operation']} {morph['best_kernel_shape']}-{morph['best_kernel_size']}",
        pred_class,
    ])

headers_sum = ["Object", "Noise", "PSNR", "MSE", "Best Seg", "IoU", "Best Morphology", "Class"]
print(tabulate(summary_rows, headers=headers_sum, tablefmt="github"))


| Object   | Noise       | PSNR     |    MSE | Best Seg   |    IoU | Best Morphology   | Class     |
|----------|-------------|----------|--------|------------|--------|-------------------|-----------|
| Circle   | salt_pepper | 32.94 dB |  33.07 | Otsu       | 0.989  | Closing ellipse-3 | Circular  |
| Wrench   | gaussian    | 30.81 dB |  53.96 | Otsu       | 0.9957 | Closing ellipse-3 | Elongated |
| Hexagon  | periodic    | 44.84 dB |   2.13 | Otsu       | 0.9985 | Closing ellipse-3 | Polygonal |
| Leaf     | gamma       | 24.50 dB | 230.88 | Global     | 0.9944 | Opening cross-7   | Irregular |
| Washer   | uniform     | 31.11 dB |  50.33 | Otsu       | 0.7721 | Closing ellipse-3 | Hollow    |


In [ ]:
import threading
import app

def run():
    demo = app.build_app()
    demo.launch(inline=True, share=False)

threading.Thread(target=run, daemon=True).start()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


Loading images...
  [✓] circle     <- real image  (images\raw\circle.png)
  [✓] wrench     <- real image  (images\raw\wrench.png)
  [✓] hexagon    <- real image  (images\raw\hexagon.png)
  [✓] leaf       <- real image  (images\raw\leaf.png)
  [✓] washer     <- real image  (images\raw\washer.png)

Ready: 5 images + masks.
Images loaded: ['circle', 'circle2', 'circle3', 'circle4', 'hexagon', 'hexagon2', 'hexagon3', 'hexagon4', 'leaf', 'leaf2', 'leaf3', 'leaf4', 'washer', 'washer2', 'washer3', 'washer4', 'wrench', 'wrench2', 'wrench3', 'wrench4']

True
